In [1]:
import enum
from itertools import count
from logging import raiseExceptions
import os
import tarfile
import pdb
import csv
import io
import copy
from scipy.stats.morestats import Std_dev
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation
from statistics import mean, stdev
import json
import time
import math
from scipy import stats
import matplotlib.cm as cm
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as patches
from scipy.spatial import distance
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from scipy.spatial import distance
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.stats import wasserstein_distance, pearsonr, spearmanr
import msgpack
import orjson
from multiprocessing import Process
from multiprocessing import Pool 
from multiprocessing import Manager

import numpy as np
import os
import pickle

# this is to do the applied-to stuff2
# note that matching and applied-to are synonyms in my language
matching_id_labels_each_tr = []
for i in range(0,17):
    matching_id_labels_each_tr.append(0)
for i in range(17,23):
    matching_id_labels_each_tr.append(1)
for i in range(23,34):
    matching_id_labels_each_tr.append(2)
for i in range(34,50):
    matching_id_labels_each_tr.append(3)
for i in range(50,66):
    matching_id_labels_each_tr.append(4)
for i in range(66,74):
    matching_id_labels_each_tr.append(5)
matching_id_labels_each_tr = np.array(matching_id_labels_each_tr)


apply_id_to_it = {2: matching_id_labels_each_tr == 2,
                  3: matching_id_labels_each_tr == 3,
                  4: matching_id_labels_each_tr == 4}

path_names = ['sameEv-sameSchema', 'sameEv-otherSchema', 
                            'otherEv-sameSchema', 'otherEv-otherSchema']

path_abbrev_dict = {"sEsS": 'sameEv-sameSchema',
                    "sEoS": 'sameEv-otherSchema',
                    "oEsS": 'otherEv-sameSchema',
                    "oEoS": 'otherEv-otherSchema'}

roi_to_focus = {"schema": ["2_2", "3_2", "4_2", "2_3", "3_3", "4_3", "2_4", "3_4", "4_4"],
                "path": ["2_2", "3_2", "4_2", "2_3", "3_3", "4_3", "2_4", "3_4", "4_4"],
                "rotated": ["2_3", "3_2"],
                "perception": ["2_2","3_3","4_4"]}

def get_perception_value_each_tr(path_to_trs):
    return (path_to_trs['sameEv-sameSchema'] + path_to_trs['sameEv-otherSchema']) \
           - (path_to_trs['otherEv-sameSchema'] + path_to_trs['otherEv-otherSchema'])
   

# 2_3 and 3_2
def get_rotated_value_each_tr(path_to_trs, focus):
    if focus == "2_3":
        return (path_to_trs['otherEv-sameSchema'] + path_to_trs['otherEv-otherSchema']) \
                - (path_to_trs['sameEv-sameSchema'] + path_to_trs['sameEv-otherSchema'])
    elif focus == "3_2":
        return (path_to_trs['otherEv-sameSchema'] + path_to_trs['sameEv-otherSchema']) \
                - (path_to_trs['sameEv-sameSchema'] + path_to_trs['otherEv-otherSchema'])

def get_path_value_each_tr(path_to_trs):
    return path_to_trs['sameEv-sameSchema'] - ( (path_to_trs['otherEv-sameSchema'] + \
                                                path_to_trs['sameEv-otherSchema'] + \
                                                path_to_trs['otherEv-otherSchema']) / 3)
   
def get_schema_value_each_tr(path_to_trs):
    return (path_to_trs['sameEv-sameSchema'] + path_to_trs['otherEv-sameSchema']) \
            - (path_to_trs['sameEv-otherSchema'] + path_to_trs['otherEv-otherSchema'])

# roi to event of the form x_y where x is the template event and y is the applied-to event
def get_roi_to_focus_to_measure(pid, wedding, template_to_pid_to_cond_to_matrices, roi_in):
    # cond for condition and path are synonyms
    focus_to_measure = {}
    this_measures_list = []
    for focus in roi_to_focus[roi_in]:
        template_id = int(focus[0])
        appliedto_id = focus[2]
        apply_it = apply_id_to_it[int(appliedto_id)]
        path_to_trs = {}
        for path in path_names:
            # get the trs for this wedding, and in the correct appliedto id
            new_trs = np.array(template_to_pid_to_cond_to_matrices[template_id][pid][path], dtype = float)[wedding,:][apply_it]
            path_to_trs[path] = new_trs
        if roi_in == "schema":
            new_measure = get_schema_value_each_tr(path_to_trs).tolist()
        elif roi_in == "perception":
            new_measure = get_perception_value_each_tr(path_to_trs).tolist()
        elif roi_in == "rotated":
            new_measure = get_rotated_value_each_tr(path_to_trs, focus).tolist()
        elif roi_in == "path":
            new_measure = get_path_value_each_tr(path_to_trs).tolist()
        else:
            print("Error: roi invalid")
            return
        focus_to_measure[focus] = new_measure
        [this_measures_list.append(x) for x in new_measure]
    return focus_to_measure, this_measures_list
            

def get_template_to_pid_to_cond_to_matrix_msgpack(light_id, in_dir = "/scratch/gpfs/rk1593/clustering_output/"):
    template_to_pid_to_cond_to_matrices = {}
    with open(in_dir + "searchlights_matrices_msgpack/" + light_id, "rb") as json_file:
        template_to_pid_to_cond_to_matrices = msgpack.load(json_file, strict_map_key = False) 
    return template_to_pid_to_cond_to_matrices

def get_template_to_pid_to_cond_to_matrix(light_id, in_dir = "/scratch/gpfs/rk1593/clustering_output/"):
    template_to_pid_to_cond_to_matrices = {}
    with open(in_dir + "searchlights_matrices_orjson/" + light_id, "rb") as f:
        template_to_pid_to_cond_to_matrices = orjson.loads(f.read())
    return template_to_pid_to_cond_to_matrices

def get_roi_to_pvalue(pid_to_roi_to_measure):
    roi_to_pvalue = {}
    for roi in roi_to_focus.keys():
        across_pids = []
        for pid in pid_to_roi_to_measure:
            across_pids.append(pid_to_roi_to_measure[pid][roi])
        tstat,pval = stats.ttest_1samp(across_pids, popmean= 0, alternative = "greater")
        roi_to_pvalue[roi] = float(pval)
    return roi_to_pvalue

def get_event_to_measure(focus_to_measure, roi, pid, wedding):
    event_list = [2,3,4] if roi != "rotated" else [2,3]
    event_to_measure = {}
    debug_bool = False
    debug_event = []
    for event in event_list:
        if roi == "schema" or roi == "path":
            avg_measure = np.nanmean(np.hstack((focus_to_measure["2_" + str(event)],
                        focus_to_measure["3_" + str(event)],
                        focus_to_measure["4_" + str(event)])))
        elif roi == "perception":
            avg_measure = np.nanmean(focus_to_measure[str(event) + "_" + str(event)])
        elif roi == "rotated":
            if event == 3:
       
                avg_measure = np.nanmean(focus_to_measure["2_3"])
            elif event == 2:
                avg_measure = np.nanmean(focus_to_measure["3_2"])
        if np.isnan(avg_measure) and pid != "sub-102":
            debug_bool = True
            debug_event.append(event)
        event_to_measure[str(event)] = float(avg_measure)
    return event_to_measure, debug_bool, debug_event

def process_chunk(light_list, output_dir):
    for light_id in light_list:
        print(light_id)
        template_to_pid_to_cond_to_matrices = get_template_to_pid_to_cond_to_matrix_msgpack(light_id)
        pid_to_roi_to_measure = {}
        pid_to_roi_to_wedding_to_event_to_measure = {}
        for pid in template_to_pid_to_cond_to_matrices[2]:
            if pid not in pid_to_roi_to_measure:
                pid_to_roi_to_measure[pid] = {}
                pid_to_roi_to_wedding_to_event_to_measure[pid] = {}
            for roi in roi_to_focus.keys():
                if roi not in pid_to_roi_to_wedding_to_event_to_measure[pid]:
                    pid_to_roi_to_wedding_to_event_to_measure[pid][roi] = {}
                measures_list = []
                for wedding in range(12):
                    # focus
                    focus_to_measure, new_measures = get_roi_to_focus_to_measure(pid, wedding,
                                            template_to_pid_to_cond_to_matrices, roi)
                    pid_to_roi_to_wedding_to_event_to_measure[pid][roi][str(wedding)], debug_bool, debug_event = get_event_to_measure(focus_to_measure, roi, pid, wedding)
                    measures_list += new_measures
                    if debug_bool and pid != "sub-102":
                        print(pid, light_id)

                        
                # this takes average across weddings and tr's and focuses
                pid_to_roi_to_measure[pid][roi] = np.nanmean(measures_list)
        roi_to_pval = get_roi_to_pvalue(pid_to_roi_to_measure)
        # light pvals
        if not os.path.exists(output_dir + light_id + "_RoiToPval"):
            with open(output_dir + light_id + "_RoiToPval", "wb") as f:
                f.write(orjson.dumps(roi_to_pval))
        # pid wedding event level neural measures
        neural_measures_path = "/scratch/gpfs/rk1593/clustering_output/searchlights_distilled_neural_measures/"
        if not os.path.exists(neural_measures_path + "each_searchlight/" + light_id):
            with open(neural_measures_path + "each_searchlight/" + light_id, "wb") as f:
                f.write(orjson.dumps(pid_to_roi_to_wedding_to_event_to_measure))

        # the way you can know if doing averages over chunk averages works 
        # as the same as aggregating than averaging is knowing
        # whether or not each chunk is the same size

def divide_chunks(l, chunk_size):
    # looping till length l
    for i in range(0, len(l), chunk_size): 
        yield l[i:i + chunk_size]

# event should be of the form x_y where x is the template event and y is the applied-to event
job_id = 62#int(os.environ["SLURM_ARRAY_TASK_ID"])
output_dir = "/scratch/gpfs/rk1593/clustering_output/brainmap_pvals/each_searchlight/"
json_path = "/scratch/gpfs/rk1593/clustering_output/jobs_info_dict_manual_jupyter_without_tuples.json"
f = open(json_path,)
jobs_info_dict = json.load(f)
this_job_searchlights = jobs_info_dict["job_id_to_searchlight_subset"][str(job_id)]
num_chunks = 2
chunk_size = math.floor(len(this_job_searchlights) / num_chunks)
chunks_of_searchlights = divide_chunks(this_job_searchlights, chunk_size = chunk_size)
processes_list = []

for index,chunk in enumerate(chunks_of_searchlights):
    process_chunk(chunk, output_dir)
#     new_p = Process(target = process_chunk, args = (chunk, output_dir))
#     new_p.start()
#     processes_list.append(new_p)

for p in processes_list:
    p.join()

54_49_43


<ipython-input-1-022df4c8e81d>:159: RuntimeWarning: Mean of empty slice
  avg_measure = np.nanmean(np.hstack((focus_to_measure["2_" + str(event)],
<ipython-input-1-022df4c8e81d>:163: RuntimeWarning: Mean of empty slice
  avg_measure = np.nanmean(focus_to_measure[str(event) + "_" + str(event)])


64_25_30
22_44_30
26_42_42
40_39_29
79_64_36
23_75_49
38_71_37
28_91_44
60_34_71
33_92_36
48_62_54
67_29_58
39_65_22
34_21_41
50_24_43
56_19_56
45_65_71
69_81_36
49_36_23
68_70_67
37_30_37
63_91_39
28_48_17
45_79_47
33_47_64
42_37_41
38_35_37
47_55_41
45_44_33
16_47_36
25_32_27
63_43_30
52_38_65
57_53_70
72_46_19
32_73_28
25_54_58
63_35_18


<ipython-input-1-022df4c8e81d>:167: RuntimeWarning: Mean of empty slice
  avg_measure = np.nanmean(focus_to_measure["2_3"])
<ipython-input-1-022df4c8e81d>:169: RuntimeWarning: Mean of empty slice
  avg_measure = np.nanmean(focus_to_measure["3_2"])


sub-134 63_35_18
sub-134 63_35_18
sub-134 63_35_18
sub-134 63_35_18
sub-134 63_35_18
sub-134 63_35_18
sub-134 63_35_18
sub-134 63_35_18
sub-136 63_35_18
sub-136 63_35_18
sub-136 63_35_18
sub-136 63_35_18
sub-136 63_35_18
sub-136 63_35_18
sub-136 63_35_18
sub-136 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
sub-141 63_35_18
61_96_51
46_25_61
60_67_32
40_70_46
40_42_36
73_29_39
47_77_67
54_25_35
52_49_70
69_84_46
26_32_27
45_64_61
72_30_60
45_73_65
24_39_66
19_58_25
63_26_60
37_101_38
69_88_42
23_52_63
58_41_12
sub-124 58_41_12
sub-124 58_

...

sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-108 69_37_11
sub-136 69_37_11
sub-136 69_37_11
sub-136 69_37_11
sub-136 69_37_11
sub-136 69_37_11
sub-136 69_37_11
sub-136 69_37_11
sub-136 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-119 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_11
sub-135 69_37_

25_32_61
72_44_70
42_33_25
63_35_28
17_61_44
26_34_23
35_56_70
69_23_40
32_60_27
44_47_39
54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-105 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80
sub-129 54_44_80


sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-110 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-114 39_38_14
sub-136 39_38_14
sub-136 39_38_14
sub-136 39_38_14
sub-136 39_38_14
sub-136 39_38_14
sub-136 39_38_14
sub-136 39_38_14
sub-136 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_14
sub-141 39_38_

sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-135 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-133 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_12
sub-128 39_33_

28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-110 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-111 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-133 28_40_12
sub-136 28_40_12
sub-136 28_40_12
sub-1

In [ ]:
#template_to_pid_to_cond_to_matrices['4']["sub-102"]['sameEv-otherSchema'][9]
# get_template_to_pid_to_cond_to_matrix_msgpack(light_id)[4]["sub-102"]['sameEv-otherSchema'][9]

In [ ]:
np.nanmean([np.nan
           ])

In [5]:
import enum
from itertools import count
from logging import raiseExceptions
import os
import tarfile
import pdb
import csv
import io
import copy
from scipy.stats.morestats import Std_dev
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from sklearn.decomposition import LatentDirichletAllocation
from statistics import mean, stdev
import json
import time
import math
from scipy import stats
import matplotlib.cm as cm
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as patches
from scipy.spatial import distance
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from scipy.spatial import distance
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.stats import wasserstein_distance, pearsonr, spearmanr
import msgpack
import orjson
from multiprocessing import Process
from multiprocessing import Pool 
from multiprocessing import Manager

import numpy as np
import os
import pickle

# this is to do the applied-to stuff2
# note that matching and applied-to are synonyms in my language
matching_id_labels_each_tr = []
for i in range(0,17):
    matching_id_labels_each_tr.append(0)
for i in range(17,23):
    matching_id_labels_each_tr.append(1)
for i in range(23,34):
    matching_id_labels_each_tr.append(2)
for i in range(34,50):
    matching_id_labels_each_tr.append(3)
for i in range(50,66):
    matching_id_labels_each_tr.append(4)
for i in range(66,74):
    matching_id_labels_each_tr.append(5)
matching_id_labels_each_tr = np.array(matching_id_labels_each_tr)


apply_id_to_it = {2: matching_id_labels_each_tr == 2,
                  3: matching_id_labels_each_tr == 3,
                  4: matching_id_labels_each_tr == 4}

path_names = ['sameEv-sameSchema', 'sameEv-otherSchema', 
                            'otherEv-sameSchema', 'otherEv-otherSchema']

path_abbrev_dict = {"sEsS": 'sameEv-sameSchema',
                    "sEoS": 'sameEv-otherSchema',
                    "oEsS": 'otherEv-sameSchema',
                    "oEoS": 'otherEv-otherSchema'}

roi_to_focus = {"schema": ["2_2", "3_2", "4_2", "2_3", "3_3", "4_3", "2_4", "3_4", "4_4"],
                "path": ["2_2", "3_2", "4_2", "2_3", "3_3", "4_3", "2_4", "3_4", "4_4"],
                "rotated": ["2_3", "3_2"],
                "perception": ["2_2","3_3","4_4"]}

roi_to_cluster_id = {"schema": 1,
                    "path":   2,
                    "rotated": 5,
                    "perception": 4}

job_id_to_roi = {0: "schema",
                 1: "path",
                 3: "rotated",
                 2: "perception"}

def get_cluster_id_to_distilled_searchlights(K, distill_numLights,
                      exaggerated_0, weight_it, cap_zero, fraction, threshold, by_sil, power_weight):
    cluster_id_to_distilled_lights = {}
    for cluster_id in range(1,K+1):
        ranks_dir = "/scratch/gpfs/rk1593/clustering_output/kmeans_searchlight_ranks/K" + str(K) + "/kmeans" + str(K) + "cluster" + str(cluster_id) + "_ranks"
        if by_sil:
            ranks_dir += "_by_silhouttes"
        else:
            if exaggerated_0:
                ranks_dir += ("_Consider0_T" + str(threshold)) 
            else:
                ranks_dir += "_NoConsider0"
            if weight_it:
                ranks_dir += ("_WeightByMean" + str(power_weight))
            else:
                ranks_dir += "_NoWeightByMean"
            if cap_zero:
                ranks_dir += "_CapZero"
            else:
                ranks_dir += "_NoCapZero"
        ranks_dir += ".csv"
        print(ranks_dir)
        ranks_df = pd.read_csv(ranks_dir)
        ranked_searchlights_in_cluster = ranks_df["searchlights_in_cluster"].tolist()
        print(len(ranked_searchlights_in_cluster))
        if fraction:
            cluster_id_to_distilled_lights[cluster_id] = ranked_searchlights_in_cluster[0:int(distill_numLights*len(ranked_searchlights_in_cluster))]
        else: 
            cluster_id_to_distilled_lights[cluster_id] = ranked_searchlights_in_cluster[0:distill_numLights]

    return cluster_id_to_distilled_lights

# event should be of the form x_y where x is the template event and y is the applied-to event
K = 5
job_id = 3 #int(os.environ["SLURM_ARRAY_TASK_ID"])
roi = job_id_to_roi[job_id]
cluster_id = roi_to_cluster_id[roi]
exaggerated_0 = False
weight_it = True
power_weight = 1
cap_zero = False
distill_it = True
distill_numLights = 0.5
fraction = True
threshold = 1.7
by_sil = False
do_json = False
# get the searchlights in this cluster
cluster_id_to_distilled_lights = get_cluster_id_to_distilled_searchlights(K, distill_numLights, exaggerated_0, weight_it, cap_zero, fraction, threshold, by_sil, power_weight)
light_list = cluster_id_to_distilled_lights[cluster_id]
output_dir = "/scratch/gpfs/rk1593/clustering_output/searchlights_distilled_neural_measures/roi_average/"
neural_measures_eachSearchlight_path = "/scratch/gpfs/rk1593/clustering_output/searchlights_distilled_neural_measures/each_searchlight/"
pid_to_wedding_to_event_to_measures_allLights = {}
avg_pid_to_wedding_to_event_to_measure = {}
# aggregate across searchlights
print(len(light_list))
for index, light_id in enumerate(light_list):
    if index % 500 == 0:
        print(index)
    light_path = neural_measures_eachSearchlight_path + light_id
    with open(light_path, "rb") as f:
        pid_to_roi_to_wedding_to_event_to_measure = orjson.loads(f.read())
    for pid in pid_to_roi_to_wedding_to_event_to_measure:
        if pid not in pid_to_wedding_to_event_to_measures_allLights:
            pid_to_wedding_to_event_to_measures_allLights[pid] = {}
            avg_pid_to_wedding_to_event_to_measure[pid] = {}
        for wedding in pid_to_roi_to_wedding_to_event_to_measure[pid][roi]:
            if wedding not in pid_to_wedding_to_event_to_measures_allLights[pid]:
                pid_to_wedding_to_event_to_measures_allLights[pid][wedding] = {}
                avg_pid_to_wedding_to_event_to_measure[pid][wedding] = {}
            for event in pid_to_roi_to_wedding_to_event_to_measure[pid][roi][wedding]:
                if event not in pid_to_wedding_to_event_to_measures_allLights[pid][wedding]:
                    pid_to_wedding_to_event_to_measures_allLights[pid][wedding][event] = []
                measure = pid_to_roi_to_wedding_to_event_to_measure[pid][roi][wedding][event]
                # turn none into nan
                if measure == None:
                    measure = np.nan
                pid_to_wedding_to_event_to_measures_allLights[pid][wedding][event] += [measure]

# take average across searchlights
for pid in pid_to_wedding_to_event_to_measures_allLights:
    for wedding in pid_to_wedding_to_event_to_measures_allLights[pid]:
        for event in pid_to_wedding_to_event_to_measures_allLights[pid][wedding]:
            avg = np.nanmean(pid_to_wedding_to_event_to_measures_allLights[pid][wedding][event])
            avg_pid_to_wedding_to_event_to_measure[pid][wedding][event] = avg
            if np.isnan(avg):
                print(pid, 'nan avg')

# output the averages across searchlights
with open(output_dir + roi + "_per_ritual_within_wedding_distilled"+ str(distill_numLights) +"_madeAfterPvalues.json", "w") as json_file:
    json.dump(avg_pid_to_wedding_to_event_to_measure, json_file)


/scratch/gpfs/rk1593/clustering_output/kmeans_searchlight_ranks/K5/kmeans5cluster1_ranks_NoConsider0_WeightByMean1_NoCapZero.csv
72008
/scratch/gpfs/rk1593/clustering_output/kmeans_searchlight_ranks/K5/kmeans5cluster2_ranks_NoConsider0_WeightByMean1_NoCapZero.csv
55531
/scratch/gpfs/rk1593/clustering_output/kmeans_searchlight_ranks/K5/kmeans5cluster3_ranks_NoConsider0_WeightByMean1_NoCapZero.csv
36091
/scratch/gpfs/rk1593/clustering_output/kmeans_searchlight_ranks/K5/kmeans5cluster4_ranks_NoConsider0_WeightByMean1_NoCapZero.csv
16980
/scratch/gpfs/rk1593/clustering_output/kmeans_searchlight_ranks/K5/kmeans5cluster5_ranks_NoConsider0_WeightByMean1_NoCapZero.csv
4795
2397
0
500
1000
1500
2000
